Implement a call transcript summarizer that outputs a structured dict: {reason_for_call, resolution, follow_up_actions, sentiment}. How would you evaluate it without ground-truth summaries at scale?

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

def summarize_transcript(transcript: str) -> dict:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"""Analyze this call transcript and return ONLY valid JSON with these fields:
- reason_for_call: string
- resolution: string (or null if unresolved)
- follow_up_actions: list of strings
- sentiment: one of "positive", "neutral", "negative", "mixed"

Transcript:
{transcript}"""
        }]
    )
    return json.loads(response.content[0].text)



In [ ]:

# Run the same transcript through the model twice with slightly different prompts and measure agreement. If sentiment flips between runs, that's a signal of ambiguity or unreliability — no human needed.

def consistency_score(transcript: str, n=3) -> float:
    results = [summarize_transcript(transcript) for _ in range(n)]
    sentiments = [r["sentiment"] for r in results]
    return sentiments.count(max(set(sentiments), key=sentiments.count)) / n

In [ ]:
# 2. LLM-as-judge (scales well)
# Have a second model critique the summary against the original transcript. It checks faithfulness, completeness, and hallucination — without needing a reference answer.

def judge_summary(transcript: str, summary: dict) -> dict:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""Does this summary accurately reflect the transcript? Score 1-5 on:
- faithfulness (no hallucinations)
- completeness (nothing important missed)
- sentiment_accuracy

Return JSON only.

Transcript: {transcript}
Summary: {json.dumps(summary)}"""
        }]
    )
    return json.loads(response.content[0].text)



A customer transcript is 6,000 tokens — too long for a single LLM context. Design a system to summarize it reliably. What are the failure modes of your approach?

[Transcript]
     |
  Split into chunks (e.g. 3 × 2000 tokens)
     |
  Summarize each chunk independently  ← "map" step
     |
  Concatenate chunk summaries
     |
  Final LLM call merges into structured output  ← "reduce" step

In [ ]:
def chunk_transcript(text: str, max_tokens: int = 2000) -> list[str]:
    """Split on speaker turns, fall back to sentences."""
    turns = text.split("\n\n")
    chunks, current = [], ""
    for turn in turns:
        if estimate_tokens(current + turn) > max_tokens:
            chunks.append(current)
            current = turn
        else:
            current += "\n\n" + turn
    if current:
        chunks.append(current)
    return chunks


def summarize_long_transcript(transcript: str) -> dict:
    chunks = chunk_transcript(transcript)

    # Map: summarize each chunk
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        resp = llm_call(
            f"Summarize this section ({i+1}/{len(chunks)}) of a call. "
            f"Note any issues raised, resolutions, and caller tone.\n\n{chunk}"
        )
        chunk_summaries.append(resp)

    # Reduce: merge into final structure
    combined = "\n---\n".join(chunk_summaries)
    return llm_call(
        f"These are summaries of consecutive sections of ONE call. "
        f"Merge into a single JSON with: reason_for_call, resolution, "
        f"follow_up_actions, sentiment.\n\n{combined}"
    )